# SAM2 Multi-Point Inference Server
Runtime → Change runtime type → **T4 GPU** → Save, then Run All.

In [ ]:
!pip install -q fastapi uvicorn python-multipart requests opencv-python-headless
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

import os

if not os.path.exists('sam2_hiera_large.pt'):
    os.system('wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt')

if not os.path.exists('cloudflared-linux-amd64'):
    os.system('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.system('chmod +x cloudflared-linux-amd64')

SUPABASE_URL = "https://base.wiserly.org/"

server_code = '''
import cv2, numpy as np, requests
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from pydantic import BaseModel
from typing import List, Optional
from PIL import Image
from io import BytesIO

SUPABASE_URL = "''' + SUPABASE_URL + '''"

app = FastAPI()
app.add_middleware(
    __import__("fastapi.middleware.cors", fromlist=["CORSMiddleware"]).CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

sam2_model = build_sam2("sam2_hiera_l.yaml", "sam2_hiera_large.pt", device="cuda")
predictor = SAM2ImagePredictor(sam2_model)
current_img_id = None
current_img_shape = None

class SegmentRequest(BaseModel):
    image_path: str
    point_coords: List[List[float]]
    point_labels: List[int]
    quality: Optional[str] = "medium"

def load_image(image_path):
    url = image_path if image_path.startswith("http") else f"{SUPABASE_URL.rstrip(\'/\')}/storage/v1/object/public/datasets/{image_path}"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    return np.array(Image.open(BytesIO(resp.content)).convert("RGB"))

def mask_to_polygon(mask, epsilon_factor=0.005):
    m8 = mask.astype(np.uint8) * 255
    _, binary = cv2.threshold(cv2.GaussianBlur(m8, (7,7), 0), 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return []
    largest = max(contours, key=cv2.contourArea)
    approx = cv2.approxPolyDP(largest, epsilon_factor * cv2.arcLength(largest, True), True)
    return approx.flatten().tolist()

@app.get("/health")
def health():
    return {"status": "ok", "model": "sam2_hiera_large", "multi_point": True}

@app.post("/segment")
async def segment(req: SegmentRequest):
    global current_img_id, current_img_shape
    if not req.point_coords or len(req.point_coords) != len(req.point_labels):
        raise HTTPException(422, "point_coords and point_labels must be equal length")
    if current_img_id != req.image_path:
        try:
            img = load_image(req.image_path)
            current_img_shape = img.shape
            predictor.set_image(img)
            current_img_id = req.image_path
        except Exception as e:
            raise HTTPException(500, f"Image load failed: {e}")
    orig_h, orig_w = current_img_shape[:2]
    coords = np.array([[np.clip(p[0],0,orig_w-1), np.clip(p[1],0,orig_h-1)] for p in req.point_coords], dtype=np.float32)
    labels = np.array(req.point_labels, dtype=np.int32)
    masks, scores, _ = predictor.predict(point_coords=coords, point_labels=labels, multimask_output=True)
    best = int(np.argmax(scores))
    return {
        "polygon": mask_to_polygon(masks[best]),
        "image_size": [orig_w, orig_h],
        "score": float(scores[best]),
        "positive_points": int(sum(labels)),
        "negative_points": int(len(labels) - sum(labels)),
    }
'''

with open('server.py', 'w') as f:
    f.write(server_code)

print('✅ Ready — run Cell 2')

## Start Server
Run each session. Copy the `trycloudflare.com` URL into your app.

In [ ]:
import threading, uvicorn, subprocess, time, re, sys
sys.path.insert(0, '.')

TUNNEL_URL = None

def start_tunnel():
    global TUNNEL_URL
    p = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8000'],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(3)
    for line in iter(p.stderr.readline, b''):
        line = line.decode('utf-8')
        if '.trycloudflare.com' in line:
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if m:
                TUNNEL_URL = m.group(0)
                print(f'\n🔥 COPY THIS URL:\n\n  {TUNNEL_URL}\n')
                break

def run_server():
    import server
    uvicorn.run(server.app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=start_tunnel, daemon=True).start()
threading.Thread(target=run_server, daemon=True).start()

try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    print('Stopped.')

## Test (Optional)

In [ ]:
import requests, json
BASE = 'http://localhost:8000'
print('Health:', json.dumps(requests.get(f'{BASE}/health').json(), indent=2))

r = requests.post(f'{BASE}/segment', json={
    'image_path': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg',
    'point_coords': [[160, 120], [10, 10]],
    'point_labels': [1, 0],
})
d = r.json()
print(f'Score: {d["score"]:.3f} | Points: {len(d["polygon"])//2} | +{d["positive_points"]} -{d["negative_points"]}')